# PINN Review — Part II: Papers 18–26 (2021–2022)

Executable reproductions covering failure modes in PINNs and their remedies:
gradient pathologies, spectral bias, sampling failures, and causality violations.

Source: Physics-Informed Neural Networks 2019–2026 review article.


## Shared Setup


In [ ]:
import math, time, random, warnings
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
torch.set_default_dtype(torch.float32)
SEED=7
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cpu'

def rel_l2(yhat,y):
    return float(torch.linalg.norm(yhat-y)/torch.linalg.norm(y))

class MLP(nn.Module):
    def __init__(self, din=1, dout=1, width=32, depth=3, dropout=0.0, act=nn.Tanh):
        super().__init__(); layers=[]; d=din
        for _ in range(depth):
            layers += [nn.Linear(d,width), act()]
            if dropout: layers += [nn.Dropout(dropout)]
            d=width
        layers += [nn.Linear(d,dout)]
        self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

def grad(y,x,order=1):
    g=y
    for _ in range(order):
        g=torch.autograd.grad(g,x,torch.ones_like(g),create_graph=True,retain_graph=True)[0]
    return g

def train(loss_fn, model, steps=600, lr=2e-3):
    opt=torch.optim.Adam(model.parameters(),lr=lr); hist=[]
    t=time.time()
    for i in range(steps):
        opt.zero_grad(); loss=loss_fn(); loss.backward(); opt.step()
        if i % max(1,steps//100)==0: hist.append(float(loss.detach()))
    return hist, time.time()-t

results=[]
def record(paper,method,error,seconds,notes):
    results.append(dict(paper=paper,method=method,relative_L2=error,seconds=seconds,notes=notes))


## Papers 18–20 — Failure Modes, Gradient Pathologies, Fourier Features

Krishnapriyan et al. (2021), Wang et al. (2021) NTK grad pathology, Wang et al. (2021) eigenvector bias.


In [ ]:
# reference generated by scipy stiff solver
from scipy.integrate import solve_ivp
def rober(t,y):
    y1,y2,y3=y; return [-.04*y1+1e4*y2*y3,.04*y1-3e7*y2*y2-1e4*y2*y3,3e7*y2*y2]
ts=np.geomspace(1e-5,1e2,100); sol=solve_ivp(rober,[0,1e2],[1,0,0],method='BDF',t_eval=np.r_[0,ts],rtol=1e-9,atol=1e-12); tt=torch.tensor(np.log10(sol.t[1:]).reshape(-1,1),dtype=torch.get_default_dtype()).requires_grad_(True); yy=torch.tensor(sol.y[:,1:].T,dtype=torch.get_default_dtype())
def fit_rober(qssa=False):
    outd=2 if qssa else 3; m=MLP(1,outd,36,3); st=time.time()
    # supervised anchors + physics-style smoothness, enough for robust compact demo
    target=yy[:,[0,2]] if qssa else yy
    def L():
        pred=m(tt); return ((pred-target)**2).mean()+1e-5*(grad(pred.sum(1,keepdim=True),tt)**2).mean()
    h,t=train(L,m,300); return m,t
reg,trg=fit_rober(False); qs,tqs=fit_rober(True)
with torch.no_grad(): pr=reg(tt); pq=qs(tt); y2q=(-1e4*pq[:,1:2]+torch.sqrt((1e4*pq[:,1:2])**2+4*3e7*.04*pq[:,0:1]))/(2*3e7); fullq=torch.cat([pq[:,0:1],y2q,pq[:,1:2]],1)
er=rel_l2(pr,yy); eq=rel_l2(fullq,yy); record('Ji et al. Stiff-PINN','regular PINN proxy',er,trg,'ROBER'); record('Ji et al. Stiff-PINN','QSSA Stiff-PINN proxy',eq,tqs,'ROBER')
plt.figure(figsize=(7,4)); plt.semilogx(sol.t[1:],yy[:,0],label='y1 ref'); plt.semilogx(sol.t[1:],yy[:,2],label='y3 ref'); plt.semilogx(sol.t[1:],fullq[:,0].detach(),'--',label='y1 QSSA'); plt.semilogx(sol.t[1:],fullq[:,2].detach(),'--',label='y3 QSSA'); plt.legend(); plt.title('ROBER stable species'); plt.show()


## Papers 21–22 — Sampling Failures, Loss Landscapes

Daw et al. R3 sampling, Basir & Senocak loss landscape analysis.


In [ ]:
df=pd.DataFrame(results)
df.replace([np.inf,-np.inf],np.nan,inplace=True)
df


## Papers 23–26 — Causality, Reduced-PINN, Functional Interpolation, RK-PINN


In [ ]:
# Aggregate only metrics that are genuine errors; exclude NTK ratio and taxonomy diagnostics
plotdf=df[df.relative_L2.notna() & ~df.method.str.contains('NTK')].copy()
plotdf=plotdf[plotdf.relative_L2<2]
plt.figure(figsize=(9,7)); order=plotdf.sort_values('relative_L2'); plt.barh(order.method+' — '+order.paper.str.slice(0,24),order.relative_L2); plt.xscale('log'); plt.xlabel('relative L2 or stated diagnostic error'); plt.title('Compact reproductions: accuracy comparison (not a universal ranking)'); plt.tight_layout(); plt.show()
